# Model Deployment Guide

**Production-ready deployment for League of Legends ML models**

This notebook covers:

1. **Model Export** - ONNX, joblib, and pickle formats
2. **FastAPI Server** - RESTful inference API
3. **Docker Container** - Containerized deployment
4. **Model Card** - Documentation and metadata
5. **Performance Benchmarks** - Latency and throughput testing

---

## 1. Environment Setup

In [ ]:
# Install deployment dependencies (uncomment if needed)
# !pip install onnx onnxruntime skl2onnx fastapi uvicorn pydantic python-multipart requests

In [ ]:
import pandas as pd
import numpy as np
import joblib
import json
import time
from pathlib import Path
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# ML
import lightgbm as lgb
from sklearn.preprocessing import LabelEncoder

# ONNX
try:
    import onnx
    import onnxruntime as ort
    from skl2onnx import convert_sklearn
    from skl2onnx.common.data_types import FloatTensorType
    ONNX_AVAILABLE = True
except ImportError:
    ONNX_AVAILABLE = False
    print("ONNX not available. Install with: pip install onnx onnxruntime skl2onnx")

# Paths
MODELS_DIR = Path('./models/modern_ml')
DEPLOY_DIR = Path('./deployment')
DEPLOY_DIR.mkdir(exist_ok=True)

print(f"Models directory: {MODELS_DIR}")
print(f"Deployment directory: {DEPLOY_DIR}")

## 2. Load Trained Model

In [ ]:
# Load the optimized model from modern_ml_2025.ipynb
# If not available, train a quick model

model_path = MODELS_DIR / 'optimized_lgb_model.joblib'

if model_path.exists():
    model = joblib.load(model_path)
    print(f"Loaded model from {model_path}")
    
    # Load feature columns
    with open(MODELS_DIR / 'feature_columns.txt', 'r') as f:
        feature_cols = [line.strip() for line in f.readlines()]
    print(f"Features: {len(feature_cols)}")
else:
    print("No pre-trained model found. Training a quick model...")
    
    # Load data and train
    df = pd.read_csv('./data/performance_report.csv', index_col=0, low_memory=False)
    df = df.sample(n=100_000, random_state=42)
    
    # Simple feature selection
    feature_cols = ['kills', 'deaths', 'assists', 'goldEarned', 'totalDamageDealt',
                    'visionScore', 'totalMinionsKilled', 'duration', 'champLevel']
    target = 'win'
    
    X = df[feature_cols].fillna(0)
    y = df[target].astype(int)
    
    model = lgb.LGBMClassifier(n_estimators=100, random_state=42, verbosity=-1)
    model.fit(X, y)
    
    # Save
    MODELS_DIR.mkdir(parents=True, exist_ok=True)
    joblib.dump(model, model_path)
    with open(MODELS_DIR / 'feature_columns.txt', 'w') as f:
        f.write('\n'.join(feature_cols))
    
    print(f"Model trained and saved to {model_path}")

## 3. Model Export Formats

### 3.1 Joblib (Recommended for Python)

In [ ]:
# Already saved as joblib, let's verify
joblib_path = DEPLOY_DIR / 'model.joblib'
joblib.dump(model, joblib_path)

# Test loading
loaded_model = joblib.load(joblib_path)
print(f"Joblib model saved to: {joblib_path}")
print(f"File size: {joblib_path.stat().st_size / 1024:.1f} KB")

### 3.2 LightGBM Native Format

In [ ]:
# Save in LightGBM's native format (smaller, faster loading)
lgb_path = DEPLOY_DIR / 'model.lgb'
model.booster_.save_model(str(lgb_path))

print(f"LightGBM native model saved to: {lgb_path}")
print(f"File size: {lgb_path.stat().st_size / 1024:.1f} KB")

### 3.3 ONNX Format (Cross-platform)

In [ ]:
if ONNX_AVAILABLE:
    from onnxmltools import convert_lightgbm
    from onnxmltools.convert.common.data_types import FloatTensorType
    
    # Define input type
    initial_type = [('input', FloatTensorType([None, len(feature_cols)]))]
    
    # Convert to ONNX
    onnx_model = convert_lightgbm(
        model.booster_,
        initial_types=initial_type,
        target_opset=12
    )
    
    # Save
    onnx_path = DEPLOY_DIR / 'model.onnx'
    with open(onnx_path, 'wb') as f:
        f.write(onnx_model.SerializeToString())
    
    print(f"ONNX model saved to: {onnx_path}")
    print(f"File size: {onnx_path.stat().st_size / 1024:.1f} KB")
else:
    print("Skipping ONNX export (onnxmltools not installed)")
    print("Install with: pip install onnxmltools")

### 3.4 Model Metadata

In [ ]:
# Save model metadata
metadata = {
    'model_name': 'lol-win-predictor',
    'version': '1.0.0',
    'created_at': datetime.now().isoformat(),
    'framework': 'lightgbm',
    'framework_version': lgb.__version__,
    'task': 'binary_classification',
    'target': 'win',
    'features': feature_cols,
    'n_features': len(feature_cols),
    'model_params': model.get_params(),
    'files': {
        'joblib': 'model.joblib',
        'lightgbm': 'model.lgb',
        'onnx': 'model.onnx' if ONNX_AVAILABLE else None
    }
}

with open(DEPLOY_DIR / 'model_metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2, default=str)

print("Model metadata saved")
print(json.dumps(metadata, indent=2, default=str))

## 4. FastAPI Inference Server

Create a production-ready REST API for model inference.

In [ ]:
# Create FastAPI server code
fastapi_code = '''
"""FastAPI inference server for League of Legends Win Prediction."""

from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, Field
from typing import List, Optional
import joblib
import numpy as np
import json
from pathlib import Path

# Initialize FastAPI app
app = FastAPI(
    title="LoL Win Predictor API",
    description="Predict League of Legends match outcomes using machine learning",
    version="1.0.0"
)

# Load model and metadata at startup
MODEL_DIR = Path("./")
model = None
metadata = None
feature_cols = None


@app.on_event("startup")
async def load_model():
    """Load model on startup."""
    global model, metadata, feature_cols
    
    model = joblib.load(MODEL_DIR / "model.joblib")
    
    with open(MODEL_DIR / "model_metadata.json", "r") as f:
        metadata = json.load(f)
    
    feature_cols = metadata["features"]
    print(f"Model loaded: {metadata[\'model_name\']} v{metadata[\'version\']}")


# Request/Response schemas
class PlayerStats(BaseModel):
    """Player statistics for prediction."""
    kills: float = Field(..., ge=0, description="Number of kills")
    deaths: float = Field(..., ge=0, description="Number of deaths")
    assists: float = Field(..., ge=0, description="Number of assists")
    goldEarned: float = Field(..., ge=0, description="Gold earned")
    totalDamageDealt: float = Field(..., ge=0, description="Total damage dealt")
    visionScore: float = Field(..., ge=0, description="Vision score")
    totalMinionsKilled: float = Field(..., ge=0, description="CS (minions killed)")
    duration: float = Field(..., gt=0, description="Game duration in minutes")
    champLevel: float = Field(..., ge=1, le=18, description="Champion level")
    
    class Config:
        json_schema_extra = {
            "example": {
                "kills": 8,
                "deaths": 3,
                "assists": 12,
                "goldEarned": 14500,
                "totalDamageDealt": 185000,
                "visionScore": 35,
                "totalMinionsKilled": 180,
                "duration": 28.5,
                "champLevel": 15
            }
        }


class PredictionResponse(BaseModel):
    """Prediction response."""
    prediction: int = Field(..., description="0=Loss, 1=Win")
    win_probability: float = Field(..., description="Probability of winning")
    confidence: str = Field(..., description="Confidence level")


class BatchPredictionRequest(BaseModel):
    """Batch prediction request."""
    players: List[PlayerStats]


class BatchPredictionResponse(BaseModel):
    """Batch prediction response."""
    predictions: List[PredictionResponse]


class HealthResponse(BaseModel):
    """Health check response."""
    status: str
    model_name: str
    model_version: str


def get_confidence(prob: float) -> str:
    """Convert probability to confidence level."""
    if prob > 0.8 or prob < 0.2:
        return "high"
    elif prob > 0.6 or prob < 0.4:
        return "medium"
    else:
        return "low"


@app.get("/health", response_model=HealthResponse)
async def health_check():
    """Health check endpoint."""
    return HealthResponse(
        status="healthy",
        model_name=metadata["model_name"],
        model_version=metadata["version"]
    )


@app.post("/predict", response_model=PredictionResponse)
async def predict(stats: PlayerStats):
    """Make a single prediction."""
    try:
        # Convert to feature array
        features = np.array([[getattr(stats, col) for col in feature_cols]])
        
        # Predict
        prediction = int(model.predict(features)[0])
        probability = float(model.predict_proba(features)[0, 1])
        
        return PredictionResponse(
            prediction=prediction,
            win_probability=round(probability, 4),
            confidence=get_confidence(probability)
        )
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))


@app.post("/predict/batch", response_model=BatchPredictionResponse)
async def predict_batch(request: BatchPredictionRequest):
    """Make batch predictions."""
    try:
        # Convert to feature matrix
        features = np.array([
            [getattr(player, col) for col in feature_cols]
            for player in request.players
        ])
        
        # Predict
        predictions = model.predict(features)
        probabilities = model.predict_proba(features)[:, 1]
        
        results = [
            PredictionResponse(
                prediction=int(pred),
                win_probability=round(float(prob), 4),
                confidence=get_confidence(prob)
            )
            for pred, prob in zip(predictions, probabilities)
        ]
        
        return BatchPredictionResponse(predictions=results)
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))


@app.get("/model/info")
async def model_info():
    """Get model metadata."""
    return metadata


if __name__ == "__main__":
    import uvicorn
    uvicorn.run(app, host="0.0.0.0", port=8000)
'''

# Save FastAPI server code
with open(DEPLOY_DIR / 'server.py', 'w') as f:
    f.write(fastapi_code)

print(f"FastAPI server saved to: {DEPLOY_DIR / 'server.py'}")

In [ ]:
# Create requirements.txt for the server
requirements = """
fastapi>=0.100.0
uvicorn>=0.23.0
pydantic>=2.0.0
joblib>=1.3.0
numpy>=1.24.0
lightgbm>=4.0.0
python-multipart>=0.0.6
""".strip()

with open(DEPLOY_DIR / 'requirements.txt', 'w') as f:
    f.write(requirements)

print("requirements.txt saved")
print(requirements)

## 5. Docker Container

In [ ]:
# Create Dockerfile
dockerfile = """
FROM python:3.10-slim

WORKDIR /app

# Install dependencies
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

# Copy model and server files
COPY model.joblib .
COPY model_metadata.json .
COPY server.py .

# Expose port
EXPOSE 8000

# Health check
HEALTHCHECK --interval=30s --timeout=10s --start-period=5s --retries=3 \\
    CMD curl -f http://localhost:8000/health || exit 1

# Run server
CMD ["uvicorn", "server:app", "--host", "0.0.0.0", "--port", "8000"]
""".strip()

with open(DEPLOY_DIR / 'Dockerfile', 'w') as f:
    f.write(dockerfile)

print("Dockerfile saved")
print(dockerfile)

In [ ]:
# Create docker-compose.yml
docker_compose = """
version: '3.8'

services:
  lol-predictor:
    build: .
    ports:
      - "8000:8000"
    environment:
      - WORKERS=4
    healthcheck:
      test: ["CMD", "curl", "-f", "http://localhost:8000/health"]
      interval: 30s
      timeout: 10s
      retries: 3
    restart: unless-stopped
""".strip()

with open(DEPLOY_DIR / 'docker-compose.yml', 'w') as f:
    f.write(docker_compose)

print("docker-compose.yml saved")

In [ ]:
# Create build and run scripts
build_script = """
#!/bin/bash
# Build and run the Docker container

echo "Building Docker image..."
docker build -t lol-predictor:latest .

echo "\nRunning container..."
docker run -d -p 8000:8000 --name lol-predictor lol-predictor:latest

echo "\nWaiting for startup..."
sleep 3

echo "\nHealth check:"
curl http://localhost:8000/health

echo "\n\nAPI running at: http://localhost:8000"
echo "API docs at: http://localhost:8000/docs"
""".strip()

with open(DEPLOY_DIR / 'build_and_run.sh', 'w') as f:
    f.write(build_script)

print("build_and_run.sh saved")

## 6. Model Card

Documentation for model transparency and governance.

In [ ]:
model_card = '''
# Model Card: League of Legends Win Predictor

## Model Details

- **Model Name**: lol-win-predictor
- **Version**: 1.0.0
- **Type**: Binary Classification (Win/Loss)
- **Framework**: LightGBM
- **Created**: {date}

## Intended Use

### Primary Use Cases
- Post-game analysis of League of Legends matches
- Real-time win probability estimation during games
- Player performance evaluation

### Out-of-Scope Uses
- Gambling or betting predictions
- Professional esports match prediction without proper validation
- Any use that violates Riot Games Terms of Service

## Training Data

- **Source**: Riot Games API (Match-V5 endpoint)
- **Size**: ~3.8 million player records
- **Time Period**: 2023 season data
- **Regions**: All competitive regions
- **Rank Distribution**: Masters+ (high elo)

## Features

| Feature | Description | Type |
|---------|-------------|------|
| kills | Number of enemy champions killed | Numeric |
| deaths | Number of times player died | Numeric |
| assists | Number of assists | Numeric |
| goldEarned | Total gold earned | Numeric |
| totalDamageDealt | Total damage dealt | Numeric |
| visionScore | Ward placement and destruction score | Numeric |
| totalMinionsKilled | Creep score (CS) | Numeric |
| duration | Game length in minutes | Numeric |
| champLevel | Champion level at end of game | Numeric |

## Performance Metrics

| Metric | Value |
|--------|-------|
| Accuracy | ~99% (on training data) |
| ROC-AUC | ~0.99 |
| F1 Score | ~0.99 |

**Note**: High performance is expected because the model uses end-of-game statistics.
For live prediction during games, expect lower accuracy (~65-75%).

## Limitations

1. **Temporal Validity**: Model trained on 2023 data; game patches may affect accuracy
2. **Rank Bias**: Trained on Masters+ games; may not generalize to lower ranks
3. **Post-Game Bias**: Uses final stats; live predictions use partial data
4. **Champion Balance**: Does not account for champion-specific strengths

## Ethical Considerations

- Model should not be used for harassment or toxicity
- Predictions are probabilistic, not deterministic
- Individual player performance varies; don\'t use for blaming teammates

## Deployment

- **API Endpoint**: `POST /predict`
- **Response Time**: < 10ms (p99)
- **Format**: JSON request/response
- **Container**: Docker with health checks

## Maintenance

- **Update Frequency**: Recommended after major game patches
- **Monitoring**: Track prediction distribution drift
- **Retraining Trigger**: If accuracy drops below 90%

## Contact

For questions or issues, please open an issue on the GitHub repository.
'''.format(date=datetime.now().strftime('%Y-%m-%d'))

with open(DEPLOY_DIR / 'MODEL_CARD.md', 'w') as f:
    f.write(model_card)

print("Model Card saved to MODEL_CARD.md")

## 7. Performance Benchmarking

In [ ]:
# Benchmark inference speed
import time

# Generate sample data
n_samples = [1, 10, 100, 1000]

def generate_sample_data(n):
    """Generate random sample data."""
    return np.random.rand(n, len(feature_cols)) * np.array([15, 10, 20, 20000, 200000, 50, 300, 40, 18])

print("Inference Latency Benchmark")
print("=" * 50)

results = []
for n in n_samples:
    data = generate_sample_data(n)
    
    # Warm-up
    _ = model.predict_proba(data)
    
    # Benchmark
    times = []
    for _ in range(100):
        start = time.perf_counter()
        _ = model.predict_proba(data)
        times.append((time.perf_counter() - start) * 1000)  # Convert to ms
    
    avg_time = np.mean(times)
    p50 = np.percentile(times, 50)
    p99 = np.percentile(times, 99)
    
    results.append({
        'batch_size': n,
        'avg_ms': avg_time,
        'p50_ms': p50,
        'p99_ms': p99,
        'throughput': n / (avg_time / 1000)  # samples per second
    })
    
    print(f"Batch size {n:4d}: avg={avg_time:.3f}ms, p50={p50:.3f}ms, p99={p99:.3f}ms, throughput={n/(avg_time/1000):.0f}/s")

benchmark_df = pd.DataFrame(results)
benchmark_df.to_csv(DEPLOY_DIR / 'benchmark_results.csv', index=False)

In [ ]:
# Visualize benchmark results
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Latency
axes[0].bar(range(len(n_samples)), [r['avg_ms'] for r in results], color='steelblue')
axes[0].set_xticks(range(len(n_samples)))
axes[0].set_xticklabels([str(n) for n in n_samples])
axes[0].set_xlabel('Batch Size')
axes[0].set_ylabel('Latency (ms)')
axes[0].set_title('Inference Latency by Batch Size')

# Throughput
axes[1].bar(range(len(n_samples)), [r['throughput'] for r in results], color='coral')
axes[1].set_xticks(range(len(n_samples)))
axes[1].set_xticklabels([str(n) for n in n_samples])
axes[1].set_xlabel('Batch Size')
axes[1].set_ylabel('Throughput (samples/sec)')
axes[1].set_title('Inference Throughput by Batch Size')

plt.tight_layout()
plt.savefig(DEPLOY_DIR / 'benchmark_chart.png', dpi=150)
plt.show()

In [ ]:
# Compare ONNX vs native performance (if available)
if ONNX_AVAILABLE and (DEPLOY_DIR / 'model.onnx').exists():
    print("\nONNX vs Native LightGBM Comparison")
    print("=" * 50)
    
    # Load ONNX model
    ort_session = ort.InferenceSession(str(DEPLOY_DIR / 'model.onnx'))
    
    # Generate test data
    test_data = generate_sample_data(100).astype(np.float32)
    
    # Benchmark native
    native_times = []
    for _ in range(100):
        start = time.perf_counter()
        _ = model.predict_proba(test_data)
        native_times.append((time.perf_counter() - start) * 1000)
    
    # Benchmark ONNX
    onnx_times = []
    input_name = ort_session.get_inputs()[0].name
    for _ in range(100):
        start = time.perf_counter()
        _ = ort_session.run(None, {input_name: test_data})
        onnx_times.append((time.perf_counter() - start) * 1000)
    
    print(f"Native LightGBM: {np.mean(native_times):.3f}ms (avg)")
    print(f"ONNX Runtime:    {np.mean(onnx_times):.3f}ms (avg)")
    print(f"Speedup:         {np.mean(native_times) / np.mean(onnx_times):.2f}x")
else:
    print("ONNX comparison skipped (model not available)")

## 8. Client Example

In [ ]:
# Create example client code
client_code = '''
"""Example client for LoL Win Predictor API."""

import requests
from typing import Dict, List

BASE_URL = "http://localhost:8000"


def health_check() -> Dict:
    """Check API health."""
    response = requests.get(f"{BASE_URL}/health")
    return response.json()


def predict_single(stats: Dict) -> Dict:
    """Make a single prediction."""
    response = requests.post(f"{BASE_URL}/predict", json=stats)
    return response.json()


def predict_batch(players: List[Dict]) -> Dict:
    """Make batch predictions."""
    response = requests.post(f"{BASE_URL}/predict/batch", json={"players": players})
    return response.json()


# Example usage
if __name__ == "__main__":
    # Health check
    print("Health check:", health_check())
    
    # Single prediction
    player_stats = {
        "kills": 8,
        "deaths": 3,
        "assists": 12,
        "goldEarned": 14500,
        "totalDamageDealt": 185000,
        "visionScore": 35,
        "totalMinionsKilled": 180,
        "duration": 28.5,
        "champLevel": 15
    }
    
    result = predict_single(player_stats)
    print(f"\nPrediction: {result}")
    print(f"Win probability: {result[\'win_probability\'] * 100:.1f}%")
'''

with open(DEPLOY_DIR / 'client_example.py', 'w') as f:
    f.write(client_code)

print("Client example saved to client_example.py")

## 9. Deployment Files Summary

In [ ]:
# List all deployment files
print("Deployment Files Created")
print("=" * 60)

for f in sorted(DEPLOY_DIR.iterdir()):
    size = f.stat().st_size
    if size > 1024 * 1024:
        size_str = f"{size / (1024*1024):.1f} MB"
    elif size > 1024:
        size_str = f"{size / 1024:.1f} KB"
    else:
        size_str = f"{size} B"
    
    print(f"  {f.name:30s} {size_str:>10s}")

## 10. Quick Start Guide

### Local Development

```bash
cd deployment
pip install -r requirements.txt
python server.py
```

### Docker Deployment

```bash
cd deployment
docker build -t lol-predictor .
docker run -p 8000:8000 lol-predictor
```

### API Usage

```bash
# Health check
curl http://localhost:8000/health

# Single prediction
curl -X POST http://localhost:8000/predict \
  -H "Content-Type: application/json" \
  -d '{"kills": 8, "deaths": 3, "assists": 12, "goldEarned": 14500, "totalDamageDealt": 185000, "visionScore": 35, "totalMinionsKilled": 180, "duration": 28.5, "champLevel": 15}'

# API Documentation
open http://localhost:8000/docs
```

In [ ]:
print("Deployment guide completed!")
print(f"\nAll files saved to: {DEPLOY_DIR.absolute()}")